In [3]:
import pandas as pd
import sqlite3

customers_url = "https://raw.githubusercontent.com/graphql-compose/graphql-compose-examples/master/examples/northwind/data/csv/customers.csv"
orders_url = "https://raw.githubusercontent.com/graphql-compose/graphql-compose-examples/master/examples/northwind/data/csv/orders.csv"

customers_df = pd.read_csv(customers_url)
orders_df = pd.read_csv(orders_url)

conn = sqlite3.connect(":memory:")
customers_df.to_sql("customers", conn, index=False, if_exists="replace")
orders_df.to_sql("orders", conn, index=False, if_exists="replace")


830

In [7]:
query = """
SELECT
    CustomerID,
    COUNT(OrderID) AS order_count,
    SUM(Freight) AS total_freight,
    AVG(Freight) AS avg_freight
FROM orders
GROUP BY CustomerID
ORDER BY total_freight DESC
"""
result = pd.read_sql_query(query, conn)

# Display top 10 rows
print(result.head(10))

  customerID  order_count  total_freight  avg_freight
0      SAVEA           31        6683.70   215.603226
1      ERNSH           30        6205.39   206.846333
2      QUICK           28        5605.63   200.201071
3      HUNGO           19        2755.24   145.012632
4      RATTC           18        2134.21   118.567222
5      QUEEN           13        1982.70   152.515385
6      FOLKO           19        1678.08    88.320000
7      BERGS           18        1559.52    86.640000
8      FRANK           15        1403.44    93.562667
9      MEREP           13        1394.22   107.247692


Using only the orders table, write a SQL query that returns each CustomerID along with:

    The total number of orders they placed (order_count)
    The total freight amount across all their orders (total_freight)
    The average freight amount per order (avg_freight)

Sort the results by total_freight in descending order.
COUNT(OrderID) → counts how many orders each customer placed

SUM(Freight) → calculates total order value (freight treated as amount)

AVG(Freight) → average order value per customer

GROUP BY CustomerID → performs aggregation per customer

ORDER BY total_freight DESC → shows highest spending customers first

In [8]:
--Query A — Using WHERE (filter before aggregation)
query_a = """
SELECT
    CustomerID,
    COUNT(OrderID) AS high_freight_orders
FROM orders
WHERE Freight > 50
GROUP BY CustomerID
"""

result_a = pd.read_sql_query(query_a, conn)
print(result_a.head())

  customerID  high_freight_orders
0      ALFKI                    2
1      ANTON                    2
2      AROUT                    2
3      BERGS                   11
4      BLAUS                    1


Only orders with Freight > 50 are kept. Then those filtered rows are grouped by CustomerID and counted. From the orders table, filter rows where Freight is greater than 50 before aggregation.

In [15]:

query_b = """
SELECT
    CustomerID,
    SUM(Freight) AS total_freight
FROM orders
GROUP BY CustomerID
HAVING SUM(Freight) > 500
"""

result_b = pd.read_sql_query(query_b, conn)
print(result_b.head())

  customerID  total_freight
0      BERGS        1559.52
1      BLONP         623.66
2      BONAP        1357.87
3      BOTTM         793.95
4      EASTC         832.34


All orders are grouped by CustomerID, the total freight per customer is calculated, and only customers whose total freight exceeds 500 are returned.

 why Query A and Query B produce different results even though both involve a threshold on Freight:

 **WHERE** filters individual rows before aggregation, so Query A only considers orders where Freight is greater than 50 and then counts them per customer.

**HAVING** filters after aggregation, so Query B first calculates each customer's total freight and then removes customers whose total freight does not exceed 500.

Therefore, Query A applies the condition to individual orders, while Query B applies the condition to aggregated customer totals, producing different results.

In [25]:

query_inner = """
SELECT
    c.CompanyName,
    c.Country,
    COUNT(o.OrderID) AS order_count,
    SUM(o.Freight) AS total_freight
FROM customers c
INNER JOIN orders o
ON c.CustomerID = o.CustomerID
GROUP BY c.CustomerID, c.CompanyName, c.Country
ORDER BY total_freight DESC
"""

result_inner = pd.read_sql_query(query_inner, conn)
print(result_inner.head())

                    companyName  country  order_count  total_freight
0            Save-a-lot Markets      USA           31        6683.70
1                  Ernst Handel  Austria           30        6205.39
2                    QUICK-Stop  Germany           28        5605.63
3  Hungry Owl All-Night Grocers  Ireland           19        2755.24
4    Rattlesnake Canyon Grocery      USA           18        2134.21


 Joins the customers and orders tables on CustomerID and returns:
This query returns only customers who have placed at least one order

In [24]:
query_left = """
SELECT
    c.CompanyName,
    c.Country,
    COUNT(o.OrderID) AS order_count,
    COALESCE(SUM(o.Freight), 0) AS total_freight
FROM customers c
LEFT JOIN orders o
ON c.CustomerID = o.CustomerID
GROUP BY c.CustomerID, c.CompanyName, c.Country
ORDER BY total_freight DESC
"""

result_left = pd.read_sql_query(query_left, conn)
print(result_left.head())

                    companyName  country  order_count  total_freight
0            Save-a-lot Markets      USA           31        6683.70
1                  Ernst Handel  Austria           30        6205.39
2                    QUICK-Stop  Germany           28        5605.63
3  Hungry Owl All-Night Grocers  Ireland           19        2755.24
4    Rattlesnake Canyon Grocery      USA           18        2134.21


Using LEFT JOIN (include all customers)
COALESCE(SUM(o.Freight), 0) converts NULL values into 0 for customers with no orders.


The **INNER JOIN** query only returns customers who have matching records in the orders table, meaning customers who have placed at least one order.

 In contrast the **LEFT JOIN** query returns all customers from the customers table, even if they have no matching orders. For customers without orders, the joined columns from the orders table become NULL, which is why COALESCE() is used to convert NULL freight values into 0.